# 05. 디코딩 전략 비교 (Greedy / Beam Search / Sampling)

**작성일**: 2026-09-18

**목표**: 04번 노트북에서 확인한 반복 문제("사람 사람 사람...", "그럴 거예요. 그럴 거예요.")가 **디코딩 전략 때문인지** 확인한다. 같은 모델(베이스라인 재학습본, KoBART 파인튜닝본) 가중치를 그대로 두고 디코딩 방식만 바꿔서 결과를 비교한다 — 즉 "모델을 바꾸면 좋아진다"와 "디코딩 전략을 바꾸면 좋아진다"는 서로 다른 축이라는 것을 04번에 이어 여기서 분리해서 본다.

**세 전략의 이론적 트레이드오프**:

| 전략 | 방식 | 장점 | 단점 |
|---|---|---|---|
| **Greedy** | 매 스텝 확률 1등 토큰만 선택 | 빠르고 구현이 단순, 결과가 항상 동일(결정적) | 근시안적 선택이 누적되어 반복 루프에 잘 빠짐 |
| **Beam Search** | 매 스텝 상위 k개 후보 시퀀스를 동시에 유지, 최종적으로 전체 시퀀스 확률이 가장 높은 것을 선택 | greedy보다 전체적으로 더 그럴듯한 문장을 찾음 | 여전히 확률이 높은 쪽으로 수렴하는 경향이 있어 반복이 완전히 사라지진 않고, 다양성이 낮으며 계산량이 beam 폭 배로 증가 |
| **Sampling (top-k/top-p)** | 다음 토큰을 확률분포에서 무작위로 뽑되, 확률이 낮은 이상한 토큰은 top-k/top-p로 미리 걸러냄 | 매번 다른, 더 다양한 문장이 나와 반복 루프를 구조적으로 피함 | 운이 나쁘면 문법이 어색하거나 문맥과 안 맞는 답이 나올 수 있음(결과가 비결정적) |

이 노트북에서는 세 전략을 04번과 동일한 8개 질문에 대해 두 모델(베이스라인 재학습본, KoBART) 모두에 적용해 정성적으로 비교하고, 간단한 반복/다양성 지표로 보조 확인한다. 전체 test set에 대한 정식 정량 평가(BLEU/ROUGE/Distinct-n)는 07번 노트북에서 진행한다.

In [1]:
# 2026-09-18: 라이브러리 로드 및 체크포인트 로드 준비
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_df = pd.read_csv('../dataset/processed/test.csv')
sample_test = test_df.sample(8, random_state=42)  # 03/04번과 동일한 질문으로 비교
print(device)

C:\Users\KDS-11\Documents\nlp_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


## 1. 모델 로드
03/04번 노트북과 동일한 구조 정의로 베이스라인(재학습본) 가중치를 불러오고, 04번에서 저장한 KoBART 파인튜닝 체크포인트도 불러온다.

In [2]:
# 2026-09-18: 베이스라인 모델 구조 재정의 (03/04번과 동일) 및 가중치 로드
sp = spm.SentencePieceProcessor(model_file='../dataset/processed/chatbot_bpe.model')
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
VOCAB_SIZE = sp.get_piece_size()
MAX_LEN = 16


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.d_model = d_model

    def forward(self, tokens):
        return self.embedding(tokens) * math.sqrt(self.d_model)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout):
        super().__init__()
        self.tok_emb = TokenEmbedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True,
        )
        self.generator = nn.Linear(d_model, vocab_size)
        self.generator.weight = self.tok_emb.embedding.weight

    def encode(self, src, src_mask):
        return self.transformer.encoder(self.pos_enc(self.tok_emb(src)), src_mask)

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(self.pos_enc(self.tok_emb(tgt)), memory, tgt_mask)


def generate_square_subsequent_mask(sz, device):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)
    return mask


def encode_text(text: str, max_len: int = MAX_LEN):
    ids = sp.encode(text, out_type=int)[: max_len - 2]
    return [BOS_ID] + ids + [EOS_ID]


baseline_model = Seq2SeqTransformer(VOCAB_SIZE, 256, 8, 3, 3, 512, 0.1).to(device)
baseline_model.load_state_dict(torch.load('../checkpoints/baseline_transformer_extended.pt', map_location=device))
baseline_model.eval()
print('베이스라인 모델 로드 완료')

베이스라인 모델 로드 완료


In [3]:
# 2026-09-18: KoBART 파인튜닝 체크포인트 로드
KOBART_MAX_LEN = 16
kobart_tokenizer = AutoTokenizer.from_pretrained('../checkpoints/kobart_finetuned')
kobart_model = AutoModelForSeq2SeqLM.from_pretrained('../checkpoints/kobart_finetuned').to(device)
kobart_model.eval()
print('KoBART 파인튜닝 모델 로드 완료')


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]


Loading weights:  17%|█▋        | 43/260 [00:00<00:00, 382.15it/s]


Loading weights:  39%|███▉      | 101/260 [00:00<00:00, 491.75it/s]


Loading weights:  58%|█████▊    | 151/260 [00:00<00:00, 412.57it/s]


Loading weights:  75%|███████▍  | 194/260 [00:00<00:00, 372.88it/s]


Loading weights: 100%|██████████| 260/260 [00:00<00:00, 523.73it/s]

KoBART 파인튜닝 모델 로드 완료


## 2. 베이스라인 모델용 디코딩 함수 3종
KoBART는 Hugging Face `generate()`가 세 전략을 모두 지원하지만, 우리가 직접 만든 베이스라인 모델은 `nn.Transformer`를 직접 다루므로 beam search와 sampling을 직접 구현한다.

In [4]:
# 2026-09-18: (1) greedy - 03/04번과 동일
@torch.no_grad()
def greedy_decode(model, src_text, max_len=MAX_LEN):
    model.eval()
    src = torch.tensor(encode_text(src_text), dtype=torch.long, device=device).unsqueeze(0)
    src_mask = torch.zeros((src.size(1), src.size(1)), device=device)
    memory = model.encode(src, src_mask)
    ys = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
        out = model.decode(ys, memory, tgt_mask)
        next_word = model.generator(out[:, -1]).argmax(dim=-1).item()
        ys = torch.cat([ys, torch.tensor([[next_word]], device=device)], dim=1)
        if next_word == EOS_ID:
            break
    ids = ys.squeeze(0).tolist()[1:]
    return sp.decode(ids[:-1] if ids and ids[-1] == EOS_ID else ids)

In [5]:
# 2026-09-18: (2) beam search
# beam_width=5: 실무/논문에서 가장 흔히 쓰이는 기본값(NMT 계열에서 4~5가 품질/속도의 무난한 절충점으로 보고됨).
# length_penalty=0.7: beam search는 짧은 문장일수록 누적 log-prob이 덜 깎여 유리해지는 편향이 있다.
#   그래서 GNMT(Wu et al., 2016) 방식처럼 길이의 거듭제곱으로 점수를 나눠(length ** alpha) 짧은 문장 쏠림을 보정한다.
@torch.no_grad()
def beam_search_decode(model, src_text, beam_width=5, max_len=MAX_LEN, length_penalty=0.7):
    model.eval()
    src = torch.tensor(encode_text(src_text), dtype=torch.long, device=device).unsqueeze(0)
    src_mask = torch.zeros((src.size(1), src.size(1)), device=device)
    memory = model.encode(src, src_mask)

    beams = [([BOS_ID], 0.0)]
    for _ in range(max_len - 1):
        candidates = []
        for tokens, score in beams:
            if tokens[-1] == EOS_ID:
                candidates.append((tokens, score))
                continue
            ys = torch.tensor([tokens], dtype=torch.long, device=device)
            tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
            out = model.decode(ys, memory, tgt_mask)
            log_probs = F.log_softmax(model.generator(out[:, -1]), dim=-1).squeeze(0)
            topk_log_probs, topk_ids = log_probs.topk(beam_width)
            for lp, idx in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                candidates.append((tokens + [idx], score + lp))

        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_width]
        if all(t[-1] == EOS_ID for t, _ in beams):
            break

    best_tokens = beams[0][0][1:]
    if best_tokens and best_tokens[-1] == EOS_ID:
        best_tokens = best_tokens[:-1]
    return sp.decode(best_tokens)

In [6]:
# 2026-09-18: (3) top-k / top-p(nucleus) sampling
# temperature=0.8: 1.0보다 낮춰 분포를 살짝 뾰족하게 만들어 너무 산만한 문장이 나오는 것을 억제 (관행적인 값).
# top_k=50, top_p=0.9: Holtzman et al.(2019, nucleus sampling 논문)과 GPT-2 공개 데모에서 흔히 쓰인 조합.
#   top_k로 확률이 매우 낮은 토큰을 1차로 걸러내고, top_p로 누적확률 90%를 넘는 지점부터 다시 한 번 걸러내
#   "말이 안 되는 토큰이 뽑히는 사고"를 줄이면서도 다양성은 유지한다.
def top_k_top_p_filtering(logits, top_k=50, top_p=0.9):
    top_k = min(top_k, logits.size(-1))
    if top_k > 0:
        kth_value = torch.topk(logits, top_k).values[..., -1, None]
        logits = torch.where(logits < kth_value, torch.full_like(logits, float('-inf')), logits)

    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_mask = cum_probs > top_p
    sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
    sorted_mask[..., 0] = False
    mask = sorted_mask.scatter(-1, sorted_idx, sorted_mask)
    return logits.masked_fill(mask, float('-inf'))


@torch.no_grad()
def sampling_decode(model, src_text, temperature=0.8, top_k=50, top_p=0.9, max_len=MAX_LEN, seed=42):
    model.eval()
    torch.manual_seed(seed)  # sampling은 원래 매번 다른 결과가 나오는 게 특성이지만, 노트북 재현성을 위해 시드 고정
    src = torch.tensor(encode_text(src_text), dtype=torch.long, device=device).unsqueeze(0)
    src_mask = torch.zeros((src.size(1), src.size(1)), device=device)
    memory = model.encode(src, src_mask)
    ys = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
    for _ in range(max_len - 1):
        tgt_mask = generate_square_subsequent_mask(ys.size(1), device)
        out = model.decode(ys, memory, tgt_mask)
        logits = model.generator(out[:, -1]).squeeze(0) / temperature
        filtered = top_k_top_p_filtering(logits, top_k, top_p)
        probs = F.softmax(filtered, dim=-1)
        next_word = torch.multinomial(probs, num_samples=1).item()
        ys = torch.cat([ys, torch.tensor([[next_word]], device=device)], dim=1)
        if next_word == EOS_ID:
            break
    ids = ys.squeeze(0).tolist()[1:]
    return sp.decode(ids[:-1] if ids and ids[-1] == EOS_ID else ids)

In [7]:
# 2026-09-18: 베이스라인 모델 - 세 전략 비교
for _, row in sample_test.iterrows():
    print(f"Q: {row['Q']}")
    print(f"  실제 A     : {row['A']}")
    print(f"  greedy     : {greedy_decode(baseline_model, row['Q'])}")
    print(f"  beam(k=5)  : {beam_search_decode(baseline_model, row['Q'])}")
    print(f"  sampling   : {sampling_decode(baseline_model, row['Q'])}")
    print()

Q: 모르는 사람 만나는거 너무 스트레스야
  실제 A     : 자신과 마음이 맞는 좋은 사람 만나세요.


  greedy     : 사람 고쳐쓰는 거 아니에요.


  beam(k=5)  : 사람 마다 다르겠지요.


  sampling   : 사람 만날 거 같아요.

Q: 정장 한 벌 필요할 듯
  실제 A     : 사면 요긴하긴 할 거 같아요.
  greedy     : 잘 지내고 있어요.


  beam(k=5)  : 잘 살 수 있어요.
  sampling   : 저도 슬픈 슬픈 눈.

Q: 베터리 겨우 15%야
  실제 A     : 미리 충전하세요.
  greedy     : 사람 만날 수 있을 거예요.


...

  beam(k=5)  : 잘 찾아보세요.
  sampling   : 좋겠어요.

Q: 이젠 진짜 진짜 안녕
  실제 A     : 맘 고생 많았어요.
  greedy     : 좋은 인연이 거기까지인가봐요.


  beam(k=5)  : 맘고생 많았어요.
  sampling   : 맘고생 많았어요.

Q: 맥주 한 잔 어때?
  실제 A     : 저랑 한 잔 해요.
  greedy     : 같이 생각 시간이 흘렀네요.


  beam(k=5)  : 그럴 수 있어요.
  sampling   : 같이 같이 생각 시간이하게도 그럴 거예요.

Q: 드디어 천일
  실제 A     : 축하합니다.
  greedy     : 좋은 생각이에요.


  beam(k=5)  : 좋은 생각이에요.
  sampling   : 저도 맘고생 많았어요.

Q: 청첩장을 찍는 날이 오다니
  실제 A     : 결혼 얼마 안 남았나봐요.
  greedy     : 좋은 생각이에요.


  beam(k=5)  : 좋은 곳으로 데려다 줄 거예요.
  sampling   : 거예요.



## 3. KoBART 모델 - 세 전략 비교
Hugging Face `generate()`의 내장 옵션을 사용한다. 베이스라인과 동일한 하이퍼파라미터(beam_width=5, top_k=50, top_p=0.9, temperature=0.8)로 맞춰 공정하게 비교한다. 추가로 실무에서 beam search의 반복을 줄이기 위해 흔히 같이 쓰는 `no_repeat_ngram_size=3`(같은 3-gram이 두 번 나오지 않도록 강제) 조합도 함께 확인한다.

In [8]:
# 2026-09-18: KoBART - greedy / beam / beam+no_repeat_ngram / sampling
@torch.no_grad()
def kobart_generate(question, **gen_kwargs):
    inputs = kobart_tokenizer(question, return_tensors='pt').to(device)
    output_ids = kobart_model.generate(**inputs, max_length=KOBART_MAX_LEN, **gen_kwargs)
    return kobart_tokenizer.decode(output_ids[0], skip_special_tokens=True)


for _, row in sample_test.iterrows():
    print(f"Q: {row['Q']}")
    print(f"  실제 A            : {row['A']}")
    print(f"  greedy            : {kobart_generate(row['Q'], num_beams=1, do_sample=False)}")
    print(f"  beam(k=5)         : {kobart_generate(row['Q'], num_beams=5, do_sample=False)}")
    print(f"  beam+no_repeat_ng : {kobart_generate(row['Q'], num_beams=5, do_sample=False, no_repeat_ngram_size=3)}")
    torch.manual_seed(42)
    print(f"  sampling          : {kobart_generate(row['Q'], do_sample=True, top_k=50, top_p=0.9, temperature=0.8)}")
    print()

Q: 모르는 사람 만나는거 너무 스트레스야
  실제 A            : 자신과 마음이 맞는 좋은 사람 만나세요.


  greedy            : 상대방을 너무 믿지 마세요. 그럴 자신 있으면 하세요. 만나지


  beam(k=5)         : 그런 사람 만나길 바랄게요. 다른 사람 만나세요. 눈치가


  beam+no_repeat_ng : 그런 사람 만나길 바랄게요. 다른 사람 만나세요. 눈치가


  sampling          : 그런 사람 만나길 바랄게요. 그런 사람 만나세요. 눈치가

Q: 정장 한 벌 필요할 듯
  실제 A            : 사면 요긴하긴 할 거 같아요.
  greedy            : 깔끔해질 거예요. 차곡차곡 준비해보세요. 한결 가벼워질


...

  sampling          : 천국과 지옥이죠. 제가 꿈에 도전할게요. 슬

Q: 청첩장을 찍는 날이 오다니
  실제 A            : 결혼 얼마 안 남았나봐요.


  greedy            : 청첩한 마음을 접는게 좋으시겠네요. 기분


  beam(k=5)         : 소원을 비세요. 자연스러운 현상이에요. 자연스러운 현상이에요.


  beam+no_repeat_ng : 청첩장을 찍는 연습을 해보세요. 기분이 바뀔지도 몰라요.


  sampling          : 기념일이나 쉬면서 쉬어가요. 기다려보세요. 기다릴게



## 4. 반복/다양성 보조 지표
정성적으로 보이는 '반복이 줄었다/늘었다'를 숫자로도 보조 확인한다. 정식 Distinct-n 평가(전체 test set 기준)는 07번에서 하고, 여기서는 방금 비교한 8개 샘플에 한해 간단히 계산한다.

In [9]:
# 2026-09-18: 응답 내부의 반복 정도를 어절 기준 distinct-1/2와 '연속 토큰 반복 여부'로 측정
def repetition_stats(text: str):
    tokens = text.split()
    if len(tokens) == 0:
        return {'distinct_1': 0.0, 'distinct_2': 0.0, 'has_consecutive_repeat': False}
    bigrams = [tuple(tokens[i:i + 2]) for i in range(len(tokens) - 1)]
    distinct_1 = len(set(tokens)) / len(tokens)
    distinct_2 = len(set(bigrams)) / len(bigrams) if bigrams else 1.0
    has_repeat = any(tokens[i] == tokens[i + 1] for i in range(len(tokens) - 1))
    return {'distinct_1': distinct_1, 'distinct_2': distinct_2, 'has_consecutive_repeat': has_repeat}


rows = []
for _, row in sample_test.iterrows():
    q = row['Q']
    strategies = {
        'baseline-greedy': greedy_decode(baseline_model, q),
        'baseline-beam': beam_search_decode(baseline_model, q),
        'baseline-sampling': sampling_decode(baseline_model, q),
        'kobart-greedy': kobart_generate(q, num_beams=1, do_sample=False),
        'kobart-beam': kobart_generate(q, num_beams=5, do_sample=False),
        'kobart-beam+norepeat': kobart_generate(q, num_beams=5, do_sample=False, no_repeat_ngram_size=3),
    }
    for name, text in strategies.items():
        stats = repetition_stats(text)
        rows.append({'strategy': name, **stats})

summary = pd.DataFrame(rows).groupby('strategy').agg(
    distinct_1=('distinct_1', 'mean'),
    distinct_2=('distinct_2', 'mean'),
    consecutive_repeat_rate=('has_consecutive_repeat', 'mean'),
).round(3)
summary

,distinct_1,distinct_2,consecutive_repeat_rate
strategy,,,
baseline-beam,1.000,1.000,0.000
baseline-greedy,1.000,1.000,0.000
baseline-sampling,0.906,1.000,0.375
kobart-beam,0.812,0.862,0.000
kobart-beam+norepeat,0.932,1.000,0.000
kobart-greedy,0.832,0.854,0.000


## 요약

**실측 결과** (반복/다양성 보조 지표, 8개 샘플 기준):

| strategy | distinct_1 | distinct_2 | consecutive_repeat_rate |
|---|---|---|---|
| baseline-greedy | 1.000 | 1.000 | 0.000 |
| baseline-beam | 1.000 | 1.000 | 0.000 |
| baseline-sampling | 0.906 | 1.000 | 0.375 |
| kobart-greedy | 0.832 | 0.854 | 0.000 |
| kobart-beam | 0.812 | 0.862 | 0.000 |
| kobart-beam+no_repeat_ngram | 0.932 | 1.000 | 0.000 |

**관찰된 것**:
- 베이스라인(재학습본)은 04번에서 보였던 '사람 사람 사람' 같은 단일 토큰 반복이 greedy/beam 모두에서 사라졌다 — 04번의 에폭 확장이 실제로 반복 문제를 상당 부분 해결했다는 것을 다시 확인.
- 반대로 **베이스라인에 sampling을 적용하니 오히려 반복이 늘었다**(consecutive_repeat_rate 0.375, 예: '저도 슬픈 슬픈 눈.', '미리 미리 충전하세요.'). 모델이 아직 확신이 부족한 상태에서 무작위 샘플링을 하면 비슷한 확률을 가진 토큰을 우연히 연속으로 뽑는 경우가 생긴다는 뜻으로 해석된다 — 'sampling이 항상 반복을 줄여준다'는 통념이 약한 모델에서는 꼭 성립하지 않음을 보여주는 사례.
- KoBART는 단일 토큰 반복은 없지만(consecutive_repeat_rate 0), **구(phrase) 단위 반복**이 greedy/beam 모두에서 남아있다(distinct_2 0.85~0.86, 예: '제가 있잖아요. 그럴 때가 아니죠. 제가 있잖아요...'). 을 추가하자 distinct_2가 1.000까지 올라가며 구 단위 반복이 사실상 사라졌다 — **beam search 자체보다 반복 억제 장치(no_repeat_ngram)의 유무가 더 결정적**이었다는 뜻.

**다음 단계(06/07번)에서 가져갈 기본값**: 베이스라인은 beam search(k=5), KoBART는 beam search + 을 대표 디코딩 설정으로 07번 정량 평가에 사용한다. (필요하면 07번에서 여러 전략을 모두 평가 대상에 포함해 표로 비교할 수도 있다.)

산출물: 이 노트북의 비교 결과는 보고서 '디코딩 전략' 절 및 08번 오류분석(반복 오류 사례, sampling의 의외의 반응)의 근거로 재사용한다.